In [1]:
!pip install transformers -q

In [3]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   DistilBERT / RoBERTa Fine-tuning                               ║
║   4-Class Text Classification                                    ║
╠══════════════════════════════════════════════════════════════════╣
║  SETUP INSTRUCTIONS:                                             ║
║                                                                  ║
║  1. Right panel → Session options → Accelerator → GPU T4 x2      ║
║  2. Right panel → Data → Upload your 3 CSV files as a dataset    ║
║  3. Update DATASET_NAME below to match your Kaggle dataset name  ║
║  4. Run All                                                      ║
╚══════════════════════════════════════════════════════════════════╝
"""

# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))


# ── Cell 3: Config — EDIT THIS SECTION ───────────────────────────────────────

# ↓↓ Change this to your Kaggle dataset name (shown in the /kaggle/input/ folder) ↓↓
DATASET_NAME = "sentimentalanalysis-dataset"

TRAIN_PATH = f"/kaggle/input/datasets/toymaker7000/sentimentalanalysis-dataset/Multilabel_Train.csv"
VAL_PATH   = f"/kaggle/input/datasets/toymaker7000/sentimentalanalysis-dataset/Multilabel_Val.csv"
TEST_PATH  = f"/kaggle/input/datasets/toymaker7000/sentimentalanalysis-dataset/Multilabel_Test - Competition.csv"
OUTPUT_DIR = "/kaggle/working"

# Model: switch to "distilbert-base-uncased" or "roberta-base"
MODEL_NAME = "roberta-base"

MAX_LEN    = 256   # used 512 
BATCH_SIZE = 8     # used 8 and 16
EPOCHS     = 8     # 5 , 6 , 8
LR         = 2e-5   # 2e-4
NUM_LABELS = 4
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing: {DEVICE}")

# Auto-detect dataset path if default doesn't work
if not os.path.exists(TRAIN_PATH):
    base = "/kaggle/input"
    for folder in os.listdir(base):
        candidate = os.path.join(base, folder, "Multilabel_Train.csv")
        if os.path.exists(candidate):
            DATASET_NAME = folder
            TRAIN_PATH = candidate
            VAL_PATH   = os.path.join(base, folder, "Multilabel_Val.csv")
            TEST_PATH  = os.path.join(base, folder, "Multilabel_Test_-_Competition.csv")
            print(f"Auto-detected dataset: {folder}")
            break
    else:
        print("ERROR: Could not find dataset. Please set DATASET_NAME manually.")


# ── Cell 4: Load & preprocess data ───────────────────────────────────────────
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].fillna("").apply(preprocess)

print(f"Train : {len(train_df):,} rows")
print(f"Val   : {len(val_df):,} rows")
print(f"Test  : {len(test_df):,} rows")
print(f"\nLabel distribution (train):\n{train_df['label'].value_counts().sort_index()}")


# ── Cell 5: Dataset & DataLoaders ────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_df["text"].tolist(), train_df["label"].tolist())
val_ds   = TextDataset(val_df["text"].tolist(),   val_df["label"].tolist())
test_ds  = TextDataset(test_df["text"].tolist())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


# ── Cell 6: Class weights ─────────────────────────────────────────────────────
classes = np.array([0, 1, 2, 3])
cw = compute_class_weight("balanced", classes=classes, y=train_df["label"].values)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f"Class weights: {dict(zip(classes, cw.round(3)))}")


# ── Cell 7: Model, optimizer, scheduler ──────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_NAME} ({total_params:.1f}M parameters)")


# ── Cell 8: Evaluation helper ─────────────────────────────────────────────────
def evaluate(loader, return_preds=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    if return_preds:
        return acc, f1, all_preds, all_labels
    return acc, f1


# ── Cell 9: Training loop ─────────────────────────────────────────────────────
print("=" * 60)
print(f"Training {MODEL_NAME} for {EPOCHS} epochs")
print("=" * 60)

best_f1    = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    acc, f1  = evaluate(val_loader)
    print(f"\nEpoch {epoch}/{EPOCHS} | Avg Loss: {avg_loss:.4f} | Val Acc: {acc:.4f} | Val F1-macro: {f1:.4f}")

    if f1 > best_f1:
        best_f1    = f1
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pt"))
        print(f"  ✓ Best model saved (F1={f1:.4f})")
    print("-" * 60)

print(f"\nBest Val F1-macro: {best_f1:.4f} at epoch {best_epoch}")


# ── Cell 10: Final evaluation with best checkpoint ───────────────────────────
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pt")))
acc, f1, val_preds, val_labels = evaluate(val_loader, return_preds=True)

print("\n" + "=" * 60)
print("FINAL VALIDATION RESULTS (best checkpoint)")
print("=" * 60)
print(f"Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"F1-macro : {f1:.4f}  ({f1*100:.2f}%)")
print()
print(classification_report(val_labels, val_preds, digits=4))


# ── Cell 11: Retrain on train+val, then predict test ─────────────────────────
print("Retraining on train + val combined for final predictions...")

all_df = pd.concat([train_df, val_df], ignore_index=True)
all_ds = TextDataset(all_df["text"].tolist(), all_df["label"].tolist())
all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

cw_all = compute_class_weight("balanced", classes=classes, y=all_df["label"].values)
class_weights_all = torch.tensor(cw_all, dtype=torch.float).to(DEVICE)
loss_fn_all = torch.nn.CrossEntropyLoss(weight=class_weights_all)

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(DEVICE)

optimizer_f = AdamW(model_final.parameters(), lr=LR, weight_decay=0.01)
total_steps_f = len(all_loader) * best_epoch
scheduler_f = get_linear_schedule_with_warmup(
    optimizer_f,
    num_warmup_steps=int(0.1 * total_steps_f),
    num_training_steps=total_steps_f,
)

for epoch in range(1, best_epoch + 1):
    model_final.train()
    total_loss = 0.0
    for batch in all_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
        optimizer_f.zero_grad()
        outputs = model_final(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn_all(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), 1.0)
        optimizer_f.step()
        scheduler_f.step()
        total_loss += loss.item()
    print(f"  Final model epoch {epoch}/{best_epoch} | Loss: {total_loss/len(all_loader):.4f}")


# ── Cell 12: Generate & save test predictions ─────────────────────────────────
model_final.eval()
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs        = model_final(input_ids=input_ids, attention_mask=attention_mask)
        preds          = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.cpu().numpy())

submission = pd.DataFrame({
    "unique_id": test_df["unique_id"],
    "label":     test_preds,
})
sub_path = os.path.join(OUTPUT_DIR, "submission_transformer.csv")
submission.to_csv(sub_path, index=False)

print(f"\nSaved: {sub_path}")
print(f"Total predictions: {len(submission)}")
print(f"\nPrediction distribution:\n{submission['label'].value_counts().sort_index()}")
print("\nDone! Download submission_transformer.csv from the Output panel →")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4

Using: cuda
Train : 2,131 rows
Val   : 711 rows
Test  : 711 rows

Label distribution (train):
label
0    1551
1     174
2     236
3     170
Name: count, dtype: int64
Train batches : 267
Val batches   : 89
Class weights: {np.int64(0): np.float64(0.343), np.int64(1): np.float64(3.062), np.int64(2): np.float64(2.257), np.int64(3): np.float64(3.134)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: roberta-base (124.6M parameters)
Training roberta-base for 8 epochs
  Epoch 1 | Step 50/267 | Loss: 1.3534
  Epoch 1 | Step 100/267 | Loss: 1.4076
  Epoch 1 | Step 150/267 | Loss: 1.3162
  Epoch 1 | Step 200/267 | Loss: 1.2573
  Epoch 1 | Step 250/267 | Loss: 1.4176

Epoch 1/8 | Avg Loss: 1.3331 | Val Acc: 0.5598 | Val F1-macro: 0.3535
  ✓ Best model saved (F1=0.3535)
------------------------------------------------------------
  Epoch 2 | Step 50/267 | Loss: 0.9016
  Epoch 2 | Step 100/267 | Loss: 1.7929
  Epoch 2 | Step 150/267 | Loss: 1.4765
  Epoch 2 | Step 200/267 | Loss: 1.6646
  Epoch 2 | Step 250/267 | Loss: 0.8320

Epoch 2/8 | Avg Loss: 1.1326 | Val Acc: 0.6428 | Val F1-macro: 0.4889
  ✓ Best model saved (F1=0.4889)
------------------------------------------------------------
  Epoch 3 | Step 50/267 | Loss: 1.0865
  Epoch 3 | Step 100/267 | Loss: 0.7957
  Epoch 3 | Step 150/267 | Loss: 0.8555
  Epoch 3 | Step 200/267 | Loss: 0.2390
  Epoch 3 | Step 250/267 | Loss: 0.240

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Final model epoch 1/3 | Loss: 1.2993
  Final model epoch 2/3 | Loss: 1.0189
  Final model epoch 3/3 | Loss: 0.7486

Saved: /kaggle/working/submission_transformer.csv
Total predictions: 711

Prediction distribution:
label
0    463
1     75
2     86
3     87
Name: count, dtype: int64

Done! Download submission_transformer.csv from the Output panel →
